# 02 - Unzip Landing Files to Bronze & 7-Day Retention Cleanup (Task 2)
Configured as **Task 2 (`unzip_to_bronze_and_cleanup`)** in a Multi-Task Databricks Workflow Job (`run_if: ALL_DONE` so it still processes queued files even if AEMO's website was temporarily unreachable in Task 1).
1. Queries `workspace.bronze.meta_file_control_log` where `status == 'DOWNLOADED'`
2. Unzips the AEMO C/I/D CSV in memory (`I == 'D'`) from `/Volumes/workspace/bronze/landing/`
3. Performs an idempotent Delta `MERGE` into `workspace.bronze.bronze_scada` on `(SETTLEMENTDATE, DUID)` using `delta.tables.DeltaTable`
4. Purges physical `.zip` files older than 7 days from `/Volumes/workspace/bronze/landing/` while preserving history in `workspace.bronze.meta_file_control_log`

### Cell 1: Imports & Unity Catalog Configuration

In [0]:
import json
import shutil
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

NEM_TZ = timezone(timedelta(hours=10))
CATALOG = "aemo"
SCHEMA = "bronze"
retention_days = 7

VOLUME_NAME = "landing/_manual_upload"
LANDING_BASE_DIR = Path(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}")
CONTROL_LOG_TABLE = f"{CATALOG}.{SCHEMA}.meta_file_control_log"
BRONZE_SCADA_TABLE = f"{CATALOG}.{SCHEMA}.scada"

SCADA_PRIMARY_KEYS = ["SETTLEMENTDATE", "DUID"]

print(f"Landing Volume (UC) : {LANDING_BASE_DIR}")
print(f"Control Log Table   : {CONTROL_LOG_TABLE}")
print(f"Bronze SCADA Table  : {BRONZE_SCADA_TABLE}")
print(f"Retention Days      : {retention_days}")

### Cell 2: Query `workspace.bronze.meta_file_control_log` for Pending (`DOWNLOADED`) ZIP Files

In [0]:
if spark.catalog.tableExists(CONTROL_LOG_TABLE):
    pending_df = (
        spark.table(CONTROL_LOG_TABLE)
        .filter("status = 'DOWNLOADED'")
        .toPandas()
    )
    print(f"Found {len(pending_df)} pending ZIP file(s) with status == 'DOWNLOADED':")
    if not pending_df.empty:
        display(pending_df)
else:
    pending_df = pd.DataFrame()
    print(f"{CONTROL_LOG_TABLE} table does not exist yet.")

### Cell 3: Unzip Pending Files in Memory & Preview Extracted SCADA Rows
Reads AEMO C/I/D CSV directly from the `.zip` archive (`skiprows=1, dtype=str`), filters to `I == 'D'`, and attaches `source_file` and `ingested_at_utc`.

In [0]:
def read_aemo_mmsdm_zip(zip_path: Path, ingested_at_utc: str) -> pd.DataFrame:
    df = pd.read_csv(zip_path, skiprows=1, dtype=str)
    df_data = df[df["I"] == "D"].copy()
    df_data["source_file"] = zip_path.name
    df_data["ingested_at_utc"] = ingested_at_utc

    if "SETTLEMENTDATE" in df_data.columns:
        df_data["settlement_date_part"] = (
            df_data["SETTLEMENTDATE"]
            .str.slice(0, 10)
            .str.replace("/", "-", regex=False)
        )
    else:
        df_data["settlement_date_part"] = "unknown"

    return df_data

now_utc = datetime.now(timezone.utc)
now_iso = now_utc.isoformat()
ingested_updates = []
batch_dfs = []

if not pending_df.empty:
    for _, row in pending_df.iterrows():
        zip_path = Path(row["landing_path"])
        if not zip_path.exists():
            print(f"[WARNING] Missing ZIP file at {zip_path}")
            continue

        df_file = read_aemo_mmsdm_zip(zip_path, ingested_at_utc=now_iso)
        batch_dfs.append(df_file)
        ingested_updates.append({
            "file_name": str(row["file_name"]),
            "status": "INGESTED_TO_BRONZE",
            "ingested_at_utc": now_iso,
            "bronze_row_count": int(len(df_file)),
        })
        print(f"Unzipped {zip_path.name}: {len(df_file):,} 'D' rows")

if batch_dfs:
    combined_df = pd.concat(batch_dfs, ignore_index=True)
    for col in combined_df.columns:
        combined_df[col] = combined_df[col].astype("string")
    print(f"\nTotal extracted rows across {len(batch_dfs)} file(s): {len(combined_df):,}")
    display(combined_df.head(10))
else:
    combined_df = pd.DataFrame()
    print("No rows extracted.")

### Cell 4: Upsert into `workspace.bronze.bronze_scada` Delta Table & Mark `INGESTED_TO_BRONZE`
Merges on `(SETTLEMENTDATE, DUID)` so overlapping intervals never create duplicate rows in Bronze.

In [0]:
total_rows_upserted = 0

if not combined_df.empty:
    available_keys = [k for k in SCADA_PRIMARY_KEYS if k in combined_df.columns]
    if available_keys:
        combined_df = combined_df.drop_duplicates(subset=available_keys, keep="last")

    sdf_batch = spark.createDataFrame(combined_df)

    if not spark.catalog.tableExists(BRONZE_SCADA_TABLE):
        print(f"Creating new {BRONZE_SCADA_TABLE} Delta table...")
        (
            sdf_batch.write.format("delta")
            .mode("overwrite")
            .partitionBy("settlement_date_part")
            .saveAsTable(BRONZE_SCADA_TABLE)
        )
    else:
        print(f"Merging into existing {BRONZE_SCADA_TABLE} Delta table...")
        dt_bronze = DeltaTable.forName(spark, BRONZE_SCADA_TABLE)
        predicate = " AND ".join([f"target.{k} = source.{k}" for k in available_keys])
        (
            dt_bronze.alias("target")
            .merge(sdf_batch.alias("source"), predicate)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    total_rows_upserted = len(combined_df)

    # Update workspace.bronze.meta_file_control_log status -> INGESTED_TO_BRONZE
    sdf_updates = spark.createDataFrame(pd.DataFrame(ingested_updates))
    dt_log = DeltaTable.forName(spark, CONTROL_LOG_TABLE)
    (
        dt_log.alias("target")
        .merge(sdf_updates.alias("source"), "target.file_name = source.file_name")
        .whenMatchedUpdate(set={
            "status": "source.status",
            "ingested_at_utc": "source.ingested_at_utc",
            "bronze_row_count": "source.bronze_row_count",
        })
        .execute()
    )
    print(f"Upserted {total_rows_upserted:,} rows to {BRONZE_SCADA_TABLE} and marked {len(ingested_updates)} file(s) as INGESTED_TO_BRONZE.")
else:
    print(f"Nothing to upsert to {BRONZE_SCADA_TABLE}.")

### Cell 5: Step 3 — Delete ZIP Files Older Than 7 Days from Unity Catalog Volume
Deletes physical `.zip` files from `/Volumes/workspace/bronze/landing/` where `status == 'INGESTED_TO_BRONZE'` and `landing_date_folder < today - 7 days`, then updates `workspace.bronze.meta_file_control_log` to `PURGED_FROM_LANDING`.

In [0]:
cutoff_date_str = (now_utc.astimezone(NEM_TZ) - timedelta(days=int(retention_days))).strftime("%Y-%m-%d")
print(f"Checking for ingested Landing ZIPs with date folder < {cutoff_date_str} ({retention_days}-day retention)...")

purged_updates = []
folders_removed = 0

if spark.catalog.tableExists(CONTROL_LOG_TABLE):
    eligible_df = (
        spark.table(CONTROL_LOG_TABLE)
        .filter(f"status = 'INGESTED_TO_BRONZE' AND landing_date_folder < '{cutoff_date_str}'")
        .toPandas()
    )

    for _, row in eligible_df.iterrows():
        zip_path = Path(row["landing_path"])
        if zip_path.exists():
            zip_path.unlink()
        purged_updates.append({
            "file_name": str(row["file_name"]),
            "status": "PURGED_FROM_LANDING",
            "purged_at_utc": now_iso,
        })

    # Remove empty date directories inside Volume
    if LANDING_BASE_DIR.exists():
        for feed_dir in LANDING_BASE_DIR.iterdir():
            if feed_dir.is_dir():
                for date_dir in feed_dir.iterdir():
                    if date_dir.is_dir() and date_dir.name < cutoff_date_str:
                        if not any(date_dir.iterdir()):
                            shutil.rmtree(date_dir, ignore_errors=True)
                            folders_removed += 1

    if purged_updates:
        sdf_purged = spark.createDataFrame(pd.DataFrame(purged_updates))
        dt_log = DeltaTable.forName(spark, CONTROL_LOG_TABLE)
        (
            dt_log.alias("target")
            .merge(sdf_purged.alias("source"), "target.file_name = source.file_name")
            .whenMatchedUpdate(set={
                "status": "source.status",
                "purged_at_utc": "source.purged_at_utc",
            })
            .execute()
        )

print(f"Purged {len(purged_updates)} old ZIP file(s) and {folders_removed} empty date folder(s).")

### Cell 6: Inspect Final `workspace.bronze.meta_file_control_log` & `workspace.bronze.bronze_scada` Summary

In [0]:
if spark.catalog.tableExists(CONTROL_LOG_TABLE):
    df_final_log = spark.table(CONTROL_LOG_TABLE).toPandas()
    print("Control Log Status Counts:")
    display(df_final_log["status"].value_counts().reset_index())
    display(df_final_log.sort_values("downloaded_at_utc", ascending=False).head(10))

result = {
    "status": "OK",
    "step_2_bronze": {"files_ingested": len(ingested_updates), "rows_upserted": total_rows_upserted},
    "step_3_cleanup": {"files_purged": len(purged_updates), "folders_removed": folders_removed},
    "timestamp_utc": now_iso,
}

try:
    dbutils.jobs.taskValues.set(key="step_2_3_summary", value=result)
    dbutils.notebook.exit(json.dumps(result))
except Exception:
    pass